# Run `train.py`

This notebook runs the repository training entry point. It is intended for a Colab or Linux runtime with a GPU.

Before running the training cell, check that `SPLIT_FILE`, `GEN_ROOT`, and `OUT_DIR` point to the right locations for your runtime.

## 1. Optional: mount Google Drive

Run this cell only in Colab if the project or data live in Drive.

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Not running in Colab, or Drive mount was skipped:', exc)

Mounted at /content/drive


## 2. Set project directory

If you are in Colab, change `PROJECT_DIR` to the folder where this repo is stored in Drive.

In [2]:
from pathlib import Path
import os

# Local default from the machine where this notebook was created.
# Colab example: Path('/content/drive/MyDrive/diffusion-segmentation')
PROJECT_DIR = Path.cwd()

os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())
print('train.py exists:', Path('train.py').exists())

Working directory: /content
train.py exists: False


## 3. Install dependencies

`monai` is used by `train.py` but is not listed in `USB/requirements.txt`, so it is installed explicitly here.

In [ ]:
%pip install -q monai nibabel scipy tqdm pandas wandb

## 4. Check GPU and paths

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

SPLIT_FILE = Path('atlas_train_val.csv')
GEN_ROOT = Path('/scratch/peirong/kxu56/USB/assets/uncond')
OUT_DIR = Path('outputs/colab_train')

print('split file:', SPLIT_FILE.resolve(), SPLIT_FILE.exists())
print('gen root:', GEN_ROOT, GEN_ROOT.exists())
print('out dir:', OUT_DIR)

# The CSV currently contains absolute /scratch/... paths. If those files are in
# Drive instead, create a remapped CSV in the next cell before training.

## 5. Optional: remap `/scratch/...` paths in the split CSV

Use this if the data were copied to another root, for example Google Drive. Edit `NEW_ROOT` first.

In [ ]:
import pandas as pd

USE_REMAP = False
OLD_ROOT = '/scratch/peirong/kxu56'
NEW_ROOT = '/content/drive/MyDrive'  # edit this to your Drive data root

if USE_REMAP:
    df = pd.read_csv(SPLIT_FILE)
    for col in ['image', 'label']:
        df[col] = df[col].astype(str).str.replace(OLD_ROOT, NEW_ROOT, regex=False)
    REMAPPED_SPLIT_FILE = Path('atlas_train_val_colab.csv')
    df.to_csv(REMAPPED_SPLIT_FILE, index=False)
    SPLIT_FILE = REMAPPED_SPLIT_FILE
    print('wrote:', SPLIT_FILE.resolve())
    print(df.head())

## 6. Run training

For a quick smoke test, set `CACHE_WORKERS=0`, `LOADER_WORKERS=0`, and consider using a tiny temporary split file.

In [ ]:
CACHE_WORKERS = 4
LOADER_WORKERS = 4
GEN_RATIO = 0.0
GEN_SEED = 40
SAVE_EVERY = 5
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

cmd = [
    'python', 'train.py',
    '--split-file', str(SPLIT_FILE),
    '--out-dir', str(OUT_DIR),
    '--gen-root', str(GEN_ROOT),
    '--gen-ratio', str(GEN_RATIO),
    '--gen-seed', str(GEN_SEED),
    '--cache-workers', str(CACHE_WORKERS),
    '--loader-workers', str(LOADER_WORKERS),
    '--device', DEVICE,
    '--save-every', str(SAVE_EVERY),
    '--show-progress',
]

print(' '.join(cmd))

In [ ]:
import subprocess

result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    raise RuntimeError(f'train.py failed with return code {result.returncode}')

## 7. Inspect outputs

In [ ]:
import json
import pandas as pd

print('Output directories:')
for path in sorted(OUT_DIR.parent.glob(OUT_DIR.name + '*')):
    print(' -', path)

summary_files = sorted(OUT_DIR.parent.glob(OUT_DIR.name + '*/run_summary.json'))
if summary_files:
    latest_summary = summary_files[-1]
    print('\nLatest summary:', latest_summary)
    print(json.dumps(json.loads(latest_summary.read_text()), indent=2))

registry = OUT_DIR.parent / 'experiment_registry.csv'
if registry.exists():
    display(pd.read_csv(registry).tail())